In [ ]:
"""
ERA5 Daily Fire Weather Index (FWI) Processor

Core functions to extract daily FWI values from ERA5 hourly NetCDF files.
Caller handles file paths, date ranges, and I/O operations.
"""

import xarray as xr
from datetime import datetime, timedelta
from pathlib import Path
from typing import Union


def extract_daily_fwi(
    file_path: Union[str, Path],
    date_label: str
) -> xr.Dataset:
    """
    Extract daily Fire Weather Index (FWI) from ERA5 hourly file.
    
    Parameters
    ----------
    file_path : str or Path
        Path to NetCDF file containing 'fwinx' variable
    date_label : str
        Date identifier (e.g., '20200115') for output dimension
    
    Returns
    -------
    xr.Dataset
        Dataset with daily FWI values (no aggregation needed—already daily)
    """
    ds = xr.open_dataset(file_path)
    
    # Extract FWI and add date dimension
    fwi_daily = ds['fwinx'].expand_dims(date=[date_label])
    
    # Package result
    result = fwi_daily.to_dataset(name='fwinx')
    result['fwinx'].attrs['units'] = 'dimensionless'
    result['fwinx'].attrs['long_name'] = 'Fire Weather Index'
    return result


def process_fwi_date_range(
    base_directory: Union[str, Path],
    start_date: datetime,
    end_date: datetime
) -> xr.Dataset:
    """
    Process consecutive days to extract daily FWI values.
    
    Parameters
    ----------
    base_directory : str or Path
        Base path containing yearly subdirectories (e.g., '.../2020/')
    start_date : datetime
        First date to process (inclusive)
    end_date : datetime
        Last date to process (exclusive)
    
    Returns
    -------
    xr.Dataset
        Concatenated daily FWI values for date range
    """
    base_dir = Path(base_directory)
    date_range = [
        start_date + timedelta(days=i)
        for i in range((end_date - start_date).days)
    ]
    
    # Process each day
    daily_datasets = [
        extract_daily_fwi(
            file_path=base_dir / str(dt.year) / f"ERA5_F_{dt.strftime('%Y%m%d')}.nc",
            date_label=dt.strftime('%Y%m%d')
        )
        for dt in date_range
    ]
    
    # Combine results
    combined = xr.concat(daily_datasets, dim='date')
    combined.attrs.update({
        'processing': 'Daily Fire Weather Index extraction',
        'source': 'ERA5 Reanalysis (Copernicus Climate Change Service)',
        'variable': 'fwinx'
    })
    return combined